<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/MF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import random

In [ ]:


content_path = "/content/sample_data/cora.content"

"""
cora.content format:

column 0 → paper ID
column 1–1433 → binary word features
last column → class label
"""

content_path = "/content/sample_data/cora.content"

cora_content = pd.read_csv(content_path, sep="\t", header=None)

paper_ids = cora_content.iloc[:, 0]
features = cora_content.iloc[:, 1:-1]
labels = cora_content.iloc[:, -1]

print("Number of papers:", len(paper_ids))
print("Feature shape:", features.shape)


Number of papers: 2708
Feature shape: (2708, 1433)


In [ ]:


"""
cora.cites format:

column 0 → cited paper
column 1 → citing paper
"""

cites_path = "/content/sample_data/cora.cites"

cora_cites = pd.read_csv(cites_path, sep="\t", header=None)
cora_cites.columns = ["cited", "citing"]

print("Number of citation edges:", len(cora_cites))


Number of citation edges: 5429


In [ ]:
"""
Map original paper IDs → integer index

Example:
31336 → 0
1061127 → 1
"""

id_map = {id_: i for i, id_ in enumerate(paper_ids)}

cora_cites["cited"] = cora_cites["cited"].map(id_map)
cora_cites["citing"] = cora_cites["citing"].map(id_map)

# Remove any rows with missing IDs
cora_cites = cora_cites.dropna()
cora_cites = cora_cites.astype(int)

In [ ]:
N = len(paper_ids)

rows = cora_cites["citing"].values
cols = cora_cites["cited"].values
data = np.ones(len(rows))

# Create sparse adjacency matrix
A = sp.coo_matrix((data, (rows, cols)), shape=(N, N))

print("Adjacency shape:", A.shape)
print("Number of edges:", A.nnz) # number of non Zero elements

Adjacency shape: (2708, 2708)
Number of edges: 5429


In [ ]:
A = A + A.T # Make undirected a
A[A > 1] = 1 # Convert 2 to 1

In [ ]:
# Convert to edge list
edges = np.array(list(zip(A.nonzero()[0], A.nonzero()[1])))

# Remove duplicate symmetric edges
edges = edges[edges[:,0] < edges[:,1]]

# Train-test split
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

In [ ]:
# Create empty adjacency
A_train = sp.lil_matrix((N, N))

for i, j in train_edges:
    A_train[i, j] = 1
    A_train[j, i] = 1

A_train = A_train.tocsr()  # converting to CSR  # Memory Efficient and Faster Multiplication

In [ ]:
k = 128  # embedding dimension
         # each node will get a vector of 128 dimensions
"""
TruncatedSVD computes top-k singular values:

A ≈ U_k Σ_k V_k^T
"""

svd = TruncatedSVD(n_components=k, random_state=42)   #approximate reconstruction
Z = svd.fit_transform(A_train)

print("Embedding shape:", Z.shape)

Embedding shape: (2708, 128)


In [ ]:
def get_scores(edges, Z):
    scores = []
    for i, j in edges:
        score = np.dot(Z[i], Z[j])
        scores.append(score)
    return np.array(scores)

In [ ]:
def generate_negative_edges(num_samples):
    neg_edges = set()

    while len(neg_edges) < num_samples:
        i = random.randint(0, N-1)
        j = random.randint(0, N-1)

        if i != j and A[i, j] == 0:
            neg_edges.add((min(i,j), max(i,j)))

    return np.array(list(neg_edges))

In [ ]:
# Positive test edges
pos_scores = get_scores(test_edges, Z)

# Negative test edges
neg_edges = generate_negative_edges(len(test_edges))
neg_scores = get_scores(neg_edges, Z)

# Combine
y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
y_scores = np.concatenate([pos_scores, neg_scores])

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)

print("Matrix Factorization AUC:", auc)

print("Matrix Factorization AP:", ap)

Matrix Factorization AUC: 0.7627177312901745
Matrix Factorization AP: 0.8207412933174096


In [ ]:
"""
==============================
Spectral / Normalized MF
==============================

We factorize:

A_norm = D^{-1/2} A D^{-1/2}

This removes degree bias.
"""

# Compute degrees
degrees = np.array(A_train.sum(axis=1)).flatten()

# D^{-1/2}
deg_inv_sqrt = np.power(degrees, -0.5)
deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0

D_inv_sqrt = sp.diags(deg_inv_sqrt) # diagonal matrix of  degree

# Normalized adjacency
A_norm = D_inv_sqrt @ A_train @ D_inv_sqrt   #Normalization helps remove the effect of high-degree nodes.

# SVD
k = 128
svd = TruncatedSVD(n_components=k, random_state=42)
Z_spec = svd.fit_transform(A_norm)

# Evaluate
pos_scores = get_scores(test_edges, Z_spec)
neg_edges = generate_negative_edges(len(test_edges))
neg_scores = get_scores(neg_edges, Z_spec)

y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
y_scores = np.concatenate([pos_scores, neg_scores])

print("Spectral MF AUC:", roc_auc_score(y_true, y_scores))
print("Spectral MF AP :", average_precision_score(y_true, y_scores))

/tmp/ipykernel_271/2515160027.py:17: RuntimeWarning: divide by zero encountered in power
  deg_inv_sqrt = np.power(degrees, -0.5)


Spectral MF AUC: 0.8037199050160699
Spectral MF AP : 0.8253359657220505


In [ ]:
"""
=====================================
Higher-Order Proximity MF
=====================================

We factorize:

S = A + A^2 + A^3

Captures multi-hop structure.
"""

A_csr = A_train.tocsr()

# Compute powers
A2 = A_csr @ A_csr
A3 = A2 @ A_csr

# Combine
S = A_csr + A2 + A3

# Normalize to avoid large values
S = S / S.max()

# SVD
k = 128
svd = TruncatedSVD(n_components=k, random_state=42)
Z_high = svd.fit_transform(S)

# Evaluate
pos_scores = get_scores(test_edges, Z_high)
neg_edges = generate_negative_edges(len(test_edges))
neg_scores = get_scores(neg_edges, Z_high)

y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
y_scores = np.concatenate([pos_scores, neg_scores])

print("Higher-Order MF AUC:", roc_auc_score(y_true, y_scores))
print("Higher-Order MF AP :", average_precision_score(y_true, y_scores))

Higher-Order MF AUC: 0.8094949853650139
Higher-Order MF AP : 0.8389109185781594


In [ ]:
"""
=====================================
PMI-based MF (DeepWalk Approximation)
=====================================

DeepWalk ≈ factorizing PMI matrix of random walks.

Steps:
1) T = D^{-1} A
2) M = T + T^2 + ... + T^k
3) Compute PMI
4) SVD
"""

# Transition matrix T = D^{-1} A
degrees = np.array(A_train.sum(axis=1)).flatten()


# Add epsilon to avoid zero division
degrees[degrees == 0] = 1e-10

deg_inv = 1.0 / degrees
D_inv = sp.diags(deg_inv)
T = D_inv @ A_train

# k-step random walk
window_size = 7

M = T.copy()
T_power = T.copy()

for _ in range(window_size - 1):
    T_power = T_power @ T
    M += T_power

M = M.toarray()

# Compute PMI
row_sums = M.sum(axis=1, keepdims=True)
col_sums = M.sum(axis=0, keepdims=True)
total_sum = M.sum()

P_ij = M / total_sum
P_i = row_sums / total_sum
P_j = col_sums / total_sum

PMI = PMI - np.log(5)
PMI[PMI < 0] = 0
# SVD
k = 256
svd = TruncatedSVD(n_components=k, random_state=42)
Z_pmi = svd.fit_transform(PMI)

# Evaluate
pos_scores = get_scores(test_edges, Z_pmi)
neg_edges = generate_negative_edges(len(test_edges))
neg_scores = get_scores(neg_edges, Z_pmi)

y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
y_scores = np.concatenate([pos_scores, neg_scores])

print("PMI-based MF AUC:", roc_auc_score(y_true, y_scores))
print("PMI-based MF AP :", average_precision_score(y_true, y_scores))

PMI-based MF AUC: 0.625422370006887
PMI-based MF AP : 0.6635169475632849


In [ ]:
"""
=========================================
PMI-based Matrix Factorization (Stable)
DeepWalk Approximation
=========================================

DeepWalk ≈ factorizing shifted PMI matrix.

Steps:
1) Compute transition matrix T = D^{-1}A
2) Compute M = T + T^2 + ... + T^window
3) Convert to probability matrix
4) Compute PMI
5) SVD
6) Evaluate
"""

# ---------------------------------------
#  Compute stable transition matrix
# ---------------------------------------

A_csr = A_train.tocsr()

row_sums = np.array(A_csr.sum(axis=1)).flatten()

# Create inverse degree safely
deg_inv = np.zeros_like(row_sums)
nonzero = row_sums > 0
deg_inv[nonzero] = 1.0 / row_sums[nonzero]

D_inv = sp.diags(deg_inv)

# Random walk transition matrix
T = D_inv @ A_csr


# ---------------------------------------
#  Compute multi-step walk matrix
# ---------------------------------------

window_size = 5

M = T.copy()
T_power = T.copy()

for _ in range(window_size - 1):
    T_power = T_power @ T
    M += T_power


# ---------------------------------------
# Convert to probability matrix
# ---------------------------------------

M = M.toarray()

total_sum = M.sum()

# Avoid division by zero
if total_sum == 0:
    total_sum = 1e-9

P_ij = M / total_sum

row_sums = P_ij.sum(axis=1, keepdims=True)
col_sums = P_ij.sum(axis=0, keepdims=True)

# Avoid zero probabilities
row_sums[row_sums == 0] = 1e-9
col_sums[col_sums == 0] = 1e-9


# ---------------------------------------
#  Compute PMI
# ---------------------------------------

PMI = np.log((P_ij + 1e-9) / (row_sums @ col_sums + 1e-9))

# DeepWalk uses Positive PMI
PMI[PMI < 0] = 0


# ---------------------------------------
# SVD Factorization
# ---------------------------------------

k = 128  # try 256 later for improvement

svd = TruncatedSVD(n_components=k, random_state=42)
Z_pmi = svd.fit_transform(PMI)


# ---------------------------------------
# Evaluate
# ---------------------------------------

pos_scores = get_scores(test_edges, Z_pmi)
neg_edges = generate_negative_edges(len(test_edges))
neg_scores = get_scores(neg_edges, Z_pmi)

y_true = np.concatenate([np.ones(len(pos_scores)),
                         np.zeros(len(neg_scores))])

y_scores = np.concatenate([pos_scores, neg_scores])

print("PMI-based MF AUC:", roc_auc_score(y_true, y_scores))
print("PMI-based MF AP :", average_precision_score(y_true, y_scores))

PMI-based MF AUC: 0.8357025510789715
PMI-based MF AP : 0.8736738114242963
